# P4 recovery — A. Seal current session
Run this notebook in your **current Colab runtime before deleting it**, using the recovery ZIP.
It reads existing Drive checkpoint files, binds their hashes to this VM boot ID and writes one origin witness.
No dependency install, model load, generation, scoring or copying checkpoints into their stores.

After both cells succeed: **Runtime → Disconnect and delete runtime**. Open notebook B in a new T4 runtime.
A kernel restart on the same VM is insufficient. Do not rerun notebook A in the new VM.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from google.colab import drive, files
SESSION_STARTED=time.monotonic()
SESSION_LIMIT_SECONDS=11.5*3600
EXPECTED_ZIP_SHA256='53ee5b56a8227d41d0c1b5a651753b11f70dcf45241849c24988b14a548202ad'
EXPECTED_MANIFEST_SHA256='e324a107b2abafe3dd7ddbc087ddd1a02c839455538da99e44c83ab4b1fdbd8f'
WORK=Path('/content/TIB_P4_RECOVERY_261020_V1')
PACKAGE=WORK/'package'
PACKAGE.mkdir(parents=True,exist_ok=True)
drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
STORE=DRIVE/'TIB_P4_RECOVERY_261020_V1'
STORE.mkdir(exist_ok=True)
WITNESS=STORE/'origin_witness.json'
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()
uploaded=files.upload()
archives=[Path(name) for name in uploaded if name.endswith('.zip')]
assert len(archives)==1,'Upload exactly the supplied recovery ZIP'
archive=archives[0]
assert sha(archive)==EXPECTED_ZIP_SHA256,'Wrong recovery ZIP'
with zipfile.ZipFile(archive) as z:
    names=z.namelist()
    assert len(names)==len(set(names)),'Duplicate members'
    for item in z.infolist():
        assert (PACKAGE/item.filename).resolve().is_relative_to(PACKAGE.resolve()),'Unsafe path'
        assert (item.external_attr>>16)&0o170000 != 0o120000,'Symlink member'
    z.extractall(PACKAGE)
assert sha(PACKAGE/'recovery_manifest.json')==EXPECTED_MANIFEST_SHA256
manifest=json.loads((PACKAGE/'recovery_manifest.json').read_text())
assert set(names)==set(manifest['files'])|{'recovery_manifest.json'}
for name,expected in manifest['files'].items():
    assert sha(PACKAGE/name)==expected,name
sys.path.insert(0,str(PACKAGE/'tools'))
from tib_p4_session_recovery import seal_origin, verify_fresh
from tib_p4_recovery_context import create_run_context, finalize
print('Recovery package verified; original stores will be read-only.')


In [ ]:
try:
    origin=seal_origin(PACKAGE,DRIVE,WITNESS)
    print(json.dumps({'state':origin['state'],'origin_session':origin['origin_session'],
                      'sealed_file_counts':{k:len(v['files']) for k,v in origin['stores'].items()},
                      'witness':str(WITNESS)},indent=2))
    print('STOP HERE. Disconnect and delete this runtime, then open notebook B with a NEW T4 runtime.')
except Exception as error:
    hold={'status':'HOLD','stage':'origin_seal','reason':str(error),'cross_session_recovery_verified':False}
    (STORE/('origin-hold-'+str(time.time_ns())+'.json')).write_text(json.dumps(hold,indent=2))
    raise
